### 1. Run Tomofast inversion fitting (steepest decent)

In [ ]:
import subprocess
import sys
import os
import time
import math
import numpy as np

# ============================================================
#  CONFIG
# ============================================================
TOMOFAST_HOME_WSL = "/home/abekhit/Tomofast-X"
WSL = r"C:\Windows\System32\wsl.exe"

base_dir     = os.getcwd()
tec_out      = os.path.join(base_dir, "Tec_files_bellon", "converted")
tomofast_dir = os.path.join(tec_out, "tomofast_forward")
forward_output_dir = os.path.join(tec_out, "Tomofast_Output")  # stage-2 forward run output (mag_prior.txt lives here)

out_dir = os.path.join(tec_out, "Tomofast_Output_Fitting_Steepest")
os.makedirs(out_dir, exist_ok=True)

print("Windows base dir:", base_dir)
print("tomofast_dir    :", tomofast_dir)
print("forward output  :", forward_output_dir)
print("out_dir         :", out_dir)

# ------------------------------------------------------------
#  USER-DEFINED START / SEARCH SETTINGS  <-- edit these
# ------------------------------------------------------------
Z_START_FRAC  = 0.25   # start depth as a fraction of grain mesh Z range (0=top, 1=bottom). 0.5 = mesh Z half
X_START_FRAC  = 0.5   # start X position as a fraction of mesh X range (0.5 = centre)
Y_START_FRAC  = 0.5   # start Y position as a fraction of mesh Y range (0.5 = centre)
MAX_STEPS     = 200   # max steepest-descent steps per grain
NUM_PROCESSES = 1      # the fitting model is always a SINGLE cell (modelGrid.size = 1 1 1) --
                        # Tomofast-X splits the model domain across MPI ranks, so a 1-cell
                        # model must run with 1 process. Using >1 here causes
                        # "Wrong dimensions in inversion_arrays_allocate_aux" / MPI_ABORT
                        # because some ranks get zero cells assigned. Do not increase this.

last_data_cost = None

# ============================================================
#  HELPERS
# ============================================================
def to_wsl_path(win_path):
    win_path = win_path.replace("\\", "/")
    drive = win_path[0].lower()
    rest  = win_path[3:]     # skip "C:/"
    return f"/mnt/{drive}/{rest}"


def run_quiet(cmd, show_error=False):
    """Run a WSL command, capture stdout/stderr, and store the last
    'data cost ... = number' value found in last_data_cost.
    If show_error=True and the command fails, print the captured output."""
    global last_data_cost
    last_data_cost = None

    p = subprocess.Popen(
        [WSL, "bash", "-lc", cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    out_lines = []
    for line in p.stdout:
        out_lines.append(line)
        if "data cost" in line.lower():
            try:
                last_data_cost = float(line.split("=")[-1].strip())
            except Exception:
                pass

    p.wait()
    if p.returncode != 0 and show_error:
        print(f"  [WSL ERROR] cmd: {cmd}")
        print("  " + "  ".join(out_lines)[-2000:])  # tail of output, avoid huge dumps
    return p.returncode, "".join(out_lines)


def read_mesh_domain(mesh_path):
    """Read a Meshgrid_<stem>.txt file (format: xmin xmax ymin ymax zmin zmax i j k
    per row) and return the overall domain bounds plus the native voxel
    cell size (dx, dy, dz)."""
    with open(mesh_path, "r") as f:
        n = int(f.readline())
        rows = np.loadtxt(f, max_rows=n)

    xmin_all, xmax_all = rows[:, 0], rows[:, 1]
    ymin_all, ymax_all = rows[:, 2], rows[:, 3]
    zmin_all, zmax_all = rows[:, 4], rows[:, 5]

    xmin, xmax = xmin_all.min(), xmax_all.max()
    ymin, ymax = ymin_all.min(), ymax_all.max()
    zmin, zmax = zmin_all.min(), zmax_all.max()

    nx = len(np.unique(np.round(xmin_all, 9)))
    ny = len(np.unique(np.round(ymin_all, 9)))
    nz = len(np.unique(np.round(zmin_all, 9)))

    dx = (xmax - xmin) / nx
    dy = (ymax - ymin) / ny
    dz = (zmax - zmin) / nz

    return dict(xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax,
                zmin=zmin, zmax=zmax, dx=dx, dy=dy, dz=dz,
                nx=nx, ny=ny, nz=nz)


# 6-neighbour offsets (+-x, +-y, +-z), in steps of the grain's native (dx, dy, dz)
NEIGH = [
    (-1, 0, 0),
    ( 1, 0, 0),
    ( 0,-1, 0),
    ( 0, 1, 0),
    ( 0, 0,-1),
    ( 0, 0, 1),
]

# ============================================================
#  ENSURE TARGET FOLDERS EXIST ON WSL
# ============================================================
forward_output_wsl = to_wsl_path(forward_output_dir)
run_quiet(f"mkdir -p {TOMOFAST_HOME_WSL}/data/TEC")
run_quiet(f"mkdir -p {TOMOFAST_HOME_WSL}/parfiles")

t0 = time.time()
best_per_grain = []

# ============================================================
#  MAIN LOOP -- one grain (stem) at a time, no blobs
# ============================================================
for stem in sorted(os.listdir(tomofast_dir)):
    body_path = os.path.join(tomofast_dir, stem)
    if not os.path.isdir(body_path):
        continue

    print("\n===================================================")
    print("Processing", stem)
    print("===================================================")

    mesh_win     = os.path.join(body_path, f"Meshgrid_{stem}.txt")
    par_win      = os.path.join(body_path, f"Parfile_{stem}_fitting.txt")
    mag_prior_win = os.path.join(forward_output_dir, stem, "data", "mag_prior.txt")

    # --------------------------------------------------------
    # 0) Pre-flight: make sure the required files actually exist
    #    (most common cause of instant "all moves invalid/NaN")
    # --------------------------------------------------------
    missing = [p for p in (mesh_win, par_win, mag_prior_win) if not os.path.exists(p)]
    if missing:
        print("  [SKIP] missing required file(s):")
        for m in missing:
            print("    -", m)
        if not os.path.exists(mag_prior_win):
            print("  -> mag_prior.txt comes from the stage-2 FORWARD run "
                  "(Parfile_{stem}.txt, not _fitting). Run that script first "
                  "for this grain so Tomofast_Output/{stem}/data/mag_prior.txt exists.")
        else:
            print("  -> re-run tec_converter.py (with the fitting-parfile snippet added) "
                  "to (re)generate this file, then re-run this script.")
        best_per_grain.append((stem, "meshgrid_NONE.txt", (float("nan"),) * 6, float("nan")))
        continue

    par_name             = f"Parfile_{stem}_fitting.txt"
    meshgrid_target_wsl  = f"{TOMOFAST_HOME_WSL}/data/TEC/Meshgrid_{stem}.txt"

    # --------------------------------------------------------
    # 1) Copy parfile + the SYNTHETIC forward-modelled field to WSL.
    #    Forward_{stem}.obs must hold the actual field values
    #    (mag_prior.txt, produced by the stage-2 forward run) --
    #    NOT the all-zero placeholder obs file -- or the data cost
    #    being minimised is meaningless (and tomofastx may error/NaN).
    # --------------------------------------------------------
    rc1, _ = run_quiet(f"cp '{to_wsl_path(par_win)}' {TOMOFAST_HOME_WSL}/parfiles/", show_error=True)
    rc2, _ = run_quiet(
        f"cp '{forward_output_wsl}/{stem}/data/mag_prior.txt' "
        f"{TOMOFAST_HOME_WSL}/data/TEC/Forward_{stem}.obs",
        show_error=True
    )
    run_quiet(f"mkdir -p {TOMOFAST_HOME_WSL}/output/TEC_{stem}_fitting")

    if rc1 != 0 or rc2 != 0:
        print(f"  [SKIP] could not copy parfile/obs to WSL for {stem} (see error above).")
        best_per_grain.append((stem, "meshgrid_NONE.txt", (float("nan"),) * 6, float("nan")))
        continue

    # --------------------------------------------------------
    # 2) Read the full grain mesh -> domain bounds + native cell size
    # --------------------------------------------------------
    dom = read_mesh_domain(mesh_win)
    dx, dy, dz = dom["dx"], dom["dy"], dom["dz"]
    nx, ny, nz = dom["nx"], dom["ny"], dom["nz"]

    # start cell as an exact mesh index (1-based, matching Meshgrid_<stem>.txt layout)
    # -- NOT a continuous coordinate -- so the fit-cell boundaries always sit exactly
    # on the same grid lines as the original mesh, never on a sensor coordinate
    # (sensors sit at cell CENTRES, i.e. half-integer offsets; grid lines are
    # integer offsets, so the two structurally never coincide).
    i0 = int(min(max(round(X_START_FRAC * nx), 1), nx))
    j0 = int(min(max(round(Y_START_FRAC * ny), 1), ny))
    k0 = int(min(max(round(Z_START_FRAC * nz), 1), nz))

    ci, cj, ck = i0, j0, k0  # current cell index (1-based)

    def bounds_from_index(i, j, k):
        xmin = dom["xmin"] + (i - 1) * dx
        xmax = dom["xmin"] + i * dx
        ymin = dom["ymin"] + (j - 1) * dy
        ymax = dom["ymin"] + j * dy
        zmin = dom["zmin"] + (k - 1) * dz
        zmax = dom["zmin"] + k * dz
        return xmin, xmax, ymin, ymax, zmin, zmax

    xmin0, xmax0, ymin0, ymax0, zmin0, zmax0 = bounds_from_index(ci, cj, ck)
    cx, cy, cz = (xmin0 + xmax0) / 2, (ymin0 + ymax0) / 2, (zmin0 + zmax0) / 2

    print(f"Domain: x[{dom['xmin']:.4f},{dom['xmax']:.4f}] "
          f"y[{dom['ymin']:.4f},{dom['ymax']:.4f}] "
          f"z[{dom['zmin']:.4f},{dom['zmax']:.4f}]  "
          f"cell=({dx:.4f},{dy:.4f},{dz:.4f})  grid=({nx},{ny},{nz})")
    print(f"Start index=({ci},{cj},{ck})  centre=({cx:.4f},{cy:.4f},{cz:.4f})  "
          f"(X_frac={X_START_FRAC}, Y_frac={Y_START_FRAC}, Z_frac={Z_START_FRAC})")

    best_cost     = float("inf")
    best_center   = (cx, cy, cz)
    best_bounds   = None
    best_meshname = "meshgrid_seed.txt"

    log_path = os.path.join(out_dir, f"{stem}_steepest_log.txt")
    with open(log_path, "w") as f:
        f.write("# step meshfile cx cy cz xmin xmax ymin ymax zmin zmax cost\n")

    tmp_mesh_win = os.path.join(base_dir, f"_meshgrid_tmp_{stem}.txt")
    tmp_mesh_wsl = to_wsl_path(tmp_mesh_win)
    step1_error_shown = False  # print only the first WSL/tomofastx failure for this grain

    # --------------------------------------------------------
    # 3) Steepest descent on the single fitting cell
    # --------------------------------------------------------
    for step in range(1, MAX_STEPS + 1):
        move_best_cost   = float("inf")
        move_best_center = None
        move_best_bounds = None
        move_best_index  = None

        for (ix, iy, iz) in NEIGH:
            ni, nj, nk = ci + ix, cj + iy, ck + iz

            # keep the fitting cell inside the original grain mesh index range
            if not (1 <= ni <= nx and 1 <= nj <= ny and 1 <= nk <= nz):
                continue

            xmin, xmax, ymin, ymax, zmin, zmax = bounds_from_index(ni, nj, nk)
            tx, ty, tz = (xmin + xmax) / 2, (ymin + ymax) / 2, (zmin + zmax) / 2

            with open(tmp_mesh_win, "w") as f:
                f.write("1\n")
                f.write(
                    f"{xmin:.6f} {xmax:.6f} {ymin:.6f} {ymax:.6f} {zmin:.6f} {zmax:.6f} "
                    "1 1 1\n"
                )

            rc, _ = run_quiet(f"cp '{tmp_mesh_wsl}' {meshgrid_target_wsl}", show_error=not step1_error_shown)
            if rc != 0:
                step1_error_shown = True
                continue

            rc, out = run_quiet(
                f"cd {TOMOFAST_HOME_WSL} && "
                f"mpirun -np {NUM_PROCESSES} ./tomofastx -p ./parfiles/{par_name}",
                show_error=not step1_error_shown
            )
            if rc != 0:
                step1_error_shown = True
                continue
            if last_data_cost is None:
                if not step1_error_shown:
                    print("  [WARN] tomofastx ran (rc=0) but no 'data cost' line was found in its output. Tail:")
                    print("  " + out[-1500:])
                    step1_error_shown = True
                continue

            cost = last_data_cost
            if math.isnan(cost) or math.isinf(cost):
                continue

            if cost < move_best_cost:
                move_best_cost   = cost
                move_best_center = (tx, ty, tz)
                move_best_bounds = (xmin, xmax, ymin, ymax, zmin, zmax)
                move_best_index  = (ni, nj, nk)

        if move_best_center is None:
            print(f"step {step:03d}: all moves invalid/NaN -> STOP")
            break

        cx, cy, cz = move_best_center
        ci, cj, ck = move_best_index
        meshname = f"meshgrid_step{step:03d}.txt"

        print(f"step {step:03d}: best cost={move_best_cost:.6e} "
              f"at center=({cx:.4f},{cy:.4f},{cz:.4f})")

        xmin, xmax, ymin, ymax, zmin, zmax = move_best_bounds
        with open(log_path, "a") as f:
            f.write(
                f"{step} {meshname} {cx:.6f} {cy:.6f} {cz:.6f} "
                f"{xmin:.6f} {xmax:.6f} {ymin:.6f} {ymax:.6f} {zmin:.6f} {zmax:.6f} "
                f"{move_best_cost:.16e}\n"
            )

        if move_best_cost >= best_cost:
            print(f"no improvement (best={best_cost:.6e}) -> STOP")
            break

        best_cost     = move_best_cost
        best_center   = (cx, cy, cz)
        best_bounds   = move_best_bounds
        best_meshname = meshname

    # --------------------------------------------------------
    # 4) Save best result, then run tomofastx ONE more time at the
    #    exact winning cell (the descent loop's last neighbour tried
    #    is not necessarily the winner, so the WSL output folder
    #    on disk does not reliably match best_bounds without this).
    # --------------------------------------------------------
    if math.isinf(best_cost) or best_bounds is None:
        best_per_grain.append((stem, "meshgrid_NONE.txt", (float("nan"),) * 6, float("nan")))
        print(f">>> BEST for {stem}: NONE -- skipping final confirmation run.")
    else:
        print(f">>> BEST for {stem}: cost={best_cost:.6e} "
              f"center=({best_center[0]:.4f},{best_center[1]:.4f},{best_center[2]:.4f})")
        best_per_grain.append((stem, best_meshname, best_bounds, best_cost))

        # write the winning single-cell mesh and push it to WSL
        xmin, xmax, ymin, ymax, zmin, zmax = best_bounds
        with open(tmp_mesh_win, "w") as f:
            f.write("1\n")
            f.write(f"{xmin:.6f} {xmax:.6f} {ymin:.6f} {ymax:.6f} {zmin:.6f} {zmax:.6f} 1 1 1\n")
        run_quiet(f"cp '{tmp_mesh_wsl}' {meshgrid_target_wsl}")

        # --------------------------------------------------------
        # 4b) FINAL CONFIRMATION RUN -- one more tomofastx run at the
        #     winning cell, so the output folder genuinely reflects
        #     the best-fit location (mirrors the combined-grain QDM run)
        # --------------------------------------------------------
        print(f"  Re-running tomofastx once more at the winning cell for {stem} ...")
        run_quiet(
            f"cd {TOMOFAST_HOME_WSL} && "
            f"{{ time mpirun -np {NUM_PROCESSES} ./tomofastx -p ./parfiles/{par_name}; }} "
            f"2>&1 | tee ./output/TEC_{stem}_fitting/log.txt",
            show_error=True
        )

        if last_data_cost is not None and abs(last_data_cost - best_cost) > 1e-6 * max(1.0, abs(best_cost)):
            print(f"  [WARN] confirmation-run cost ({last_data_cost:.6e}) does not match "
                  f"the descent's recorded best cost ({best_cost:.6e}). Using the confirmation "
                  f"run's output as the authoritative result.")

        # --------------------------------------------------------
        # 4c) Copy output -- same structure as the QDM combined-grain run:
        #     data, model, Paraview, log, Parfile_copy.txt, costs.txt
        # --------------------------------------------------------
        output_win = os.path.join(out_dir, stem)
        os.makedirs(output_win, exist_ok=True)
        output_wsl = to_wsl_path(output_win)
        wsl_grain_out = f"{TOMOFAST_HOME_WSL}/output/TEC_{stem}_fitting"

        run_quiet(f"cp -r {wsl_grain_out}/data '{output_wsl}/'", show_error=True)
        run_quiet(f"cp -r {wsl_grain_out}/model '{output_wsl}/'", show_error=True)
        run_quiet(f"cp -r {wsl_grain_out}/Paraview '{output_wsl}/'", show_error=True)
        run_quiet(f"cp -r {wsl_grain_out}/log.txt '{output_wsl}/'", show_error=True)
        run_quiet(f"cp -r {wsl_grain_out}/Parfile_copy.txt '{output_wsl}/'", show_error=True)
        run_quiet(f"cp -r {wsl_grain_out}/costs.txt '{output_wsl}/'", show_error=True)
        # run_quiet(f"cp -r {wsl_grain_out}/SENSIT '{output_wsl}/'")  # optional, very large

        # also save the winning 1-cell mesh alongside the output -- downstream
        # analysis (per-cell magnetization/moment scripts) needs a mesh file
        # with the SAME row count as mag_final_model_full.txt (1 row here),
        # not the full multi-cell Meshgrid_{stem}.txt from tomofast_forward/
        winning_mesh_path = os.path.join(output_win, f"Meshgrid_{stem}_fitting.txt")
        with open(winning_mesh_path, "w") as f:
            f.write("1\n")
            f.write(f"{xmin:.6f} {xmax:.6f} {ymin:.6f} {ymax:.6f} {zmin:.6f} {zmax:.6f} 1 1 1\n")
        print(f"  Saved winning 1-cell mesh: {winning_mesh_path}")

    # clean up temp mesh file
    if os.path.exists(tmp_mesh_win):
        os.remove(tmp_mesh_win)

# ============================================================
#  WRITE SUMMARY -- best_per_grain_steepest.txt
# ============================================================
summary_path = os.path.join(out_dir, "best_per_grain_steepest.txt")
with open(summary_path, "w") as f:
    f.write("# grain meshfile xmin xmax ymin ymax zmin zmax final_data_cost\n")
    for (stem, meshname, b, cost) in best_per_grain:
        xmin, xmax, ymin, ymax, zmin, zmax = b
        f.write(
            f"{stem} {meshname} "
            f"{xmin:.6f} {xmax:.6f} {ymin:.6f} {ymax:.6f} {zmin:.6f} {zmax:.6f} "
            f"{cost:.16e}\n"
        )

print("\nWrote:", summary_path)
print(f"\nTotal runtime: {(time.time() - t0) / 60:.2f} minutes")
print("\nAll done.")

### 2. Plot Tomofast Output (Bxyz misfits)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

base_dir = os.getcwd()
tomofast_out = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_Fitting_Steepest")
print("Reading from:", tomofast_out)

# Column count -> {component_name: column_index}
#   3 cols: x y Bz       4 cols: x y z Bz       6+ cols: x y z Bx By Bz
COLUMN_LAYOUTS = {3: {"Bz": 2}, 4: {"Bz": 3}}


def get_components(ncols):
    if ncols in COLUMN_LAYOUTS:
        return COLUMN_LAYOUTS[ncols]
    if ncols >= 6:
        return {"Bx": 3, "By": 4, "Bz": 5}
    return None


def load_2d(path):
    """Load a data file, guaranteeing a 2D array even if it has only one row."""
    data = np.loadtxt(path, skiprows=1)
    return data.reshape(1, -1) if data.ndim == 1 else data


def plot_bxyz(x_coords, y_coords, data, sfx, out_dir, label):
    os.makedirs(out_dir, exist_ok=True)

    components = get_components(data.shape[1])
    if components is None:
        print(f"[SKIP] {sfx} {label}: unsupported number of columns = {data.shape[1]}")
        return

    fig, axes = plt.subplots(1, len(components), figsize=(6 * len(components), 6))
    if len(components) == 1:
        axes = [axes]
    fig.suptitle(f"{sfx} \u2014 {label}", fontsize=14)

    bz_col = data[:, components["Bz"]]
    scale = np.nanmax(np.abs(bz_col))

    for ax, (component, col) in zip(axes, components.items()):
        b_values = -data[:, col] if component == "Bz" else data[:, col]  # flip only Bz, tomofast -> qdm convention
        df = pd.DataFrame({"x": x_coords, "y": y_coords, "b": b_values})
        df = df.sort_values(["y", "x"]).reset_index(drop=True)
        unique_x = np.sort(df["x"].unique())
        unique_y = np.sort(df["y"].unique())
        nx, ny = len(unique_x), len(unique_y)
        if len(df) != nx * ny:
            print(f"[SKIP] {sfx} {label} {component}: not a regular grid")
            continue

        X = df["x"].values.reshape(ny, nx)
        Y = df["y"].values.reshape(ny, nx)
        Z = df["b"].values.reshape(ny, nx)

        im = ax.contourf(X, Y, Z, levels=100, cmap="seismic", vmin=-scale, vmax=scale)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(f"{component} (nT)", fontsize=12)
        cbar.ax.tick_params(labelsize=11)
        ax.set_xlabel("x ($\\mu$m)", fontsize=12)
        ax.set_ylabel("y ($\\mu$m)", fontsize=12)
        ax.set_title(component, fontsize=13)
        ax.tick_params(labelsize=11)
        ax.set_aspect("equal")
        ax.ticklabel_format(useOffset=False, style="plain")
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))

    plt.tight_layout()
    filename = os.path.join(out_dir, f"Bxyz_{sfx}_{label}.png")
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved:", filename)


for sfx in sorted(os.listdir(tomofast_out)):
    sample_dir = os.path.join(tomofast_out, sfx)
    if not os.path.isdir(sample_dir):
        continue

    data_dir = os.path.join(sample_dir, "data")
    files_to_plot = {
        "observed": os.path.join(data_dir, "mag_observed.txt"),
        "calculated": os.path.join(data_dir, "mag_final.txt"),
        "misfit": os.path.join(data_dir, "mag_misfit.txt"),
    }

    for label, filepath in files_to_plot.items():
        if not os.path.isfile(filepath):
            print(f"[missing] {filepath}")
            continue

        data = load_2d(filepath)
        x_coords = data[:, 0]
        y_coords = data[:, 1]
        plot_bxyz(x_coords, y_coords, data, sfx, sample_dir, label)

print("\nDone.")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

base_dir = os.getcwd()
tomofast_out = os.path.join(
    base_dir,
    "Tec_files_bellon",
    "converted",
    "Tomofast_Output_Fitting_Steepest"
)

print("Reading from:", tomofast_out)


def get_components_from_columns(ncols):
    """
    Detect file format based on number of columns.

    Supported formats:
    x y Bz              = 3 columns
    x y z Bz            = 4 columns
    x y z Bx By Bz      = 6 columns
    """

    if ncols == 3:
        return {"Bz": 2}

    elif ncols == 4:
        return {"Bz": 3}

    elif ncols >= 6:
        return {
            "Bx": 3,
            "By": 4,
            "Bz": 5,
        }

    else:
        return None


def plot_component_obs_calc_misfit(sfx, sample_dir):
    os.makedirs(sample_dir, exist_ok=True)

    data_dir = os.path.join(sample_dir, "data")

    files_to_plot = {
        "observed": os.path.join(data_dir, "mag_observed.txt"),
        "calculated": os.path.join(data_dir, "mag_final.txt"),
        "misfit": os.path.join(data_dir, "mag_misfit.txt"),
    }

    data_all = {}

    for label, filepath in files_to_plot.items():
        if not os.path.isfile(filepath):
            print(f"[missing] {filepath}")
            return

        data = np.loadtxt(filepath, skiprows=1)

        if data.ndim == 1:
            data = data.reshape(1, -1)

        data_all[label] = data

    ncols = data_all["observed"].shape[1]
    components = get_components_from_columns(ncols)

    if components is None:
        print(f"[SKIP] {sfx}: unsupported number of columns = {ncols}")
        return

    print(f"{sfx}: detected {ncols} columns -> plotting {list(components.keys())}")

    for component, col in components.items():
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle(f"{sfx} - {component}", fontsize=14)

        obs_values = data_all["observed"][:, col]

        # Flip only Bz, tomofast -> qdm convention
        if component == "Bz":
            obs_values = -obs_values

        scale = np.nanmax(np.abs(obs_values))

        for ax, label in zip(axes, ["observed", "calculated", "misfit"]):
            data = data_all[label]

            if data.shape[1] <= col:
                print(f"[SKIP] {sfx} {component} {label}: column {col} missing")
                continue

            x_coords = data[:, 0]
            y_coords = data[:, 1]

            b_values = data[:, col]

            # Flip only Bz, tomofast -> qdm convention
            if component == "Bz":
                b_values = -b_values

            df = pd.DataFrame({
                "x": x_coords,
                "y": y_coords,
                "b": b_values
            })

            df = df.sort_values(["y", "x"]).reset_index(drop=True)

            unique_x = np.sort(df["x"].unique())
            unique_y = np.sort(df["y"].unique())

            nx, ny = len(unique_x), len(unique_y)

            if len(df) != nx * ny:
                print(f"[SKIP] {sfx} {component} {label}: not a regular grid")
                continue

            X = df["x"].values.reshape(ny, nx)
            Y = df["y"].values.reshape(ny, nx)
            Z = df["b"].values.reshape(ny, nx)

            im = ax.contourf(
                X,
                Y,
                Z,
                levels=100,
                cmap="seismic",
                vmin=-scale,
                vmax=scale
            )

            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label(f"{component} (nT)", fontsize=12)
            cbar.ax.tick_params(labelsize=11)

            ax.set_xlabel("x (µm)", fontsize=12)
            ax.set_ylabel("y (µm)", fontsize=12)
            ax.set_title(label, fontsize=13)
            ax.tick_params(labelsize=11)
            ax.set_aspect("equal")
            ax.ticklabel_format(useOffset=False, style="plain")
            ax.xaxis.set_major_locator(plt.MaxNLocator(4))
            ax.yaxis.set_major_locator(plt.MaxNLocator(4))

        plt.tight_layout()

        filename = os.path.join(
            sample_dir,
            f"{component}_obs_calc_misfit_{sfx}.png"
        )

        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close()

        print("Saved:", filename)


for sfx in sorted(os.listdir(tomofast_out)):
    sample_dir = os.path.join(tomofast_out, sfx)

    if not os.path.isdir(sample_dir):
        continue

    plot_component_obs_calc_misfit(sfx, sample_dir)


print("\nDone.")

### 3- Inversion metrics

In [ ]:
# ------------------------------------------------------------
# Compute inversion fit metrics (data cost, R^2, RMS) per body,
# comparing observed vs. calculated (final model) field data.
# ------------------------------------------------------------
import os
import numpy as np
import pandas as pd

base_dir = os.getcwd()
tomofast_out = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_Fitting_Steepest")
print("Reading from:", tomofast_out)

# Column count -> {component_name: column_index}, covers all 3 file formats:
#   3 cols: x y Bz          4 cols: x y z Bz          6+ cols: x y z Bx By Bz
COLUMN_LAYOUTS = {3: {"Bz": 2}, 4: {"Bz": 3}}


def get_components(ncols):
    if ncols in COLUMN_LAYOUTS:
        return COLUMN_LAYOUTS[ncols]
    if ncols >= 6:
        return {"Bx": 3, "By": 4, "Bz": 5}
    return None


def load_2d(path):
    """Load a data file, guaranteeing a 2D array even if it has only one row."""
    data = np.loadtxt(path, skiprows=1)
    return data.reshape(1, -1) if data.ndim == 1 else data


def calc_metrics(obs, calc):
    """
    Data cost (relative L2 norm) and R^2 between observed and calculated field values.
    NOTE: with these definitions, RMS (root-mean-square residual, normalized by the
    RMS of obs) is mathematically identical to data_cost for any data - the n's
    cancel out (sqrt(mean(r^2))/sqrt(mean(o^2)) == norm(r)/norm(o)). It's returned
    separately only for clarity/readability in the output table, not as an
    independent check. If you want a true (non-normalized) RMS instead, change
    `rms = np.sqrt(np.mean(residuals ** 2))` and drop the division by obs_rms.
    """
    residuals = calc - obs

    data_cost = np.linalg.norm(residuals) / np.linalg.norm(obs)
    #rms = np.sqrt(np.mean(residuals ** 2)) / np.sqrt(np.mean(obs ** 2))
    rms = np.sqrt(np.mean(residuals ** 2))

    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((obs - obs.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

    return data_cost, r2, rms


results = []

for sfx in sorted(os.listdir(tomofast_out)):
    sample_dir = os.path.join(tomofast_out, sfx)
    if not os.path.isdir(sample_dir):
        continue

    obs_file = os.path.join(sample_dir, "data", "mag_observed.txt")
    calc_file = os.path.join(sample_dir, "data", "mag_final.txt")
    if not os.path.isfile(obs_file) or not os.path.isfile(calc_file):
        print(f"[missing obs/calc] {sfx}")
        continue

    obs_data = load_2d(obs_file)
    calc_data = load_2d(calc_file)

    components = get_components(obs_data.shape[1])
    if components is None:
        print(f"[SKIP] {sfx}: unsupported column count = {obs_data.shape[1]}")
        continue

    print(f"{sfx}: detected {obs_data.shape[1]} columns -> calculating {list(components.keys())}")

    for component, col in components.items():
        if calc_data.shape[1] <= col:
            print(f"[SKIP] {sfx} {component}: calculated file missing column {col}")
            continue

        obs_values = obs_data[:, col]
        calc_values = calc_data[:, col]
        data_cost, r2, rms = calc_metrics(obs_values, calc_values)

        results.append({
            "inversion": sfx,
            "component": component,
            "n_data": len(obs_values),
            "data_cost": data_cost,
            "r2": r2,
            "rms": rms,
        })

metrics_df = pd.DataFrame(results)
print("\nMetrics:")
print(metrics_df)

out_csv = os.path.join(tomofast_out, "inversion_metrics.csv")
metrics_df.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)
print("\nDone.")

### 4- Calculate parameters of the inverted models

In [ ]:
# ------------------------------------------------------------
# Per-cell parameters from Tomofast Inverted magnetization + mesh
# LOOP over all body_* folders -> ONE nonzero-only CSV per body
#
# FITTED (single-cell) model version: reads the WINNING 1-cell mesh
# saved alongside the steepest-descent fitting output
# (Meshgrid_{stem}_fitting.txt), not the full multi-cell
# Meshgrid_{stem}.txt from tomofast_forward/ -- that file has one row
# per voxel in the original mesh, while the fitted model has exactly
# ONE row (the winning dipole cell), so they don't line up.
# ------------------------------------------------------------

import os
import glob
import numpy as np
import pandas as pd

base_dir = os.getcwd()
TOMOFAST_ROOT_In  = os.path.join(base_dir, "Tec_files_bellon", "converted", "tomofast_forward")
TOMOFAST_ROOT_Out = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_Fitting_Steepest")

UM_TO_M = 1e-6


# ============================================================
# Magnetization -> Dec, Inc, |M|
#
#   |M|   = sqrt(Mx^2 + My^2 + Mz^2)
#   Inc   = arccos(-Mz / |M|) - 90
#           (Mz negated: Tomofast's internal z-axis is positive-down,
#            micromag/QDM convention is positive-up)
#   Dec   = 90 - atan2(My, Mx)
# ============================================================
def dec_inc_norm(Mx, My, Mz):
    Mx, My, Mz = np.atleast_1d(Mx, My, Mz)
    norm = np.sqrt(Mx**2 + My**2 + Mz**2)
    Mz_used = -Mz
    safe_norm = np.where(norm > 0, norm, np.nan)
    theta = np.degrees(np.arccos(np.clip(Mz_used / safe_norm, -1, 1)))
    phi = np.degrees(np.arctan2(My, Mx))
    inc = theta - 90.0
    dec = np.mod(90.0 - phi, 360.0)
    return dec, inc, norm


body_dirs = sorted(
    d for d in os.listdir(TOMOFAST_ROOT_Out)
    if d.startswith("body_") and os.path.isdir(os.path.join(TOMOFAST_ROOT_Out, d))
)
print("Found bodies:", body_dirs)

for body in body_dirs:
    print("\n===================================================")
    print("Processing", body)
    print("===================================================")

    body_folder_Out = os.path.join(TOMOFAST_ROOT_Out, body)

    mag_matches = glob.glob(os.path.join(body_folder_Out, "model", "mag_final_model_full.txt"))

    # the winning 1-cell mesh, saved alongside the fitting output --
    # NOT the full multi-cell mesh from tomofast_forward/
    mesh_matches = glob.glob(os.path.join(body_folder_Out, f"Meshgrid_{body}_fitting.txt"))

    if not mag_matches or not mesh_matches:
        print("   Missing fitted Magnetization or fitted (1-cell) Meshgrid file - skipping.")
        if not mesh_matches:
            print(f"   -> expected: {os.path.join(body_folder_Out, f'Meshgrid_{body}_fitting.txt')}")
            print("      (re-run the steepest-descent fitting script with the latest "
                  "version that saves this file)")
        continue

    mesh = pd.read_csv(mesh_matches[0], sep=r"\s+", skiprows=1, header=None,
                        names=["X1", "X2", "Y1", "Y2", "Z1", "Z2", "i", "j", "k"])
    mag = pd.read_csv(mag_matches[0], sep=r"\s+", skiprows=1, header=None,
                       names=["Mx", "My", "Mz"])

    if len(mesh) != len(mag):
        print(f"   Cell count mismatch ({len(mesh)} vs {len(mag)}) - skipping.")
        continue

    df = pd.concat([mesh, mag], axis=1)
    df.insert(0, "CellID", np.arange(1, len(df) + 1))
    print(f"   Loaded {len(df)} cell(s) (fitted single-cell model).")

    # ------------------------------------------------------------
    # Cell geometry and volume
    #   dx = X2-X1, dy = Y2-Y1, dz = Z2-Z1   (converted um -> m)
    #   V  = dx * dy * dz
    # ------------------------------------------------------------
    df["dx"] = (df["X2"] - df["X1"]) * UM_TO_M
    df["dy"] = (df["Y2"] - df["Y1"]) * UM_TO_M
    df["dz"] = (df["Z2"] - df["Z1"]) * UM_TO_M
    df["Volume"] = df["dx"] * df["dy"] * df["dz"]

    # ------------------------------------------------------------
    # Direction and magnitude (see dec_inc_norm above),
    # from the INVERTED (fitted) magnetization
    # ------------------------------------------------------------
    df["Dec"], df["Inc"], df["Mag_norm"] = dec_inc_norm(df["Mx"], df["My"], df["Mz"])

    # ------------------------------------------------------------
    # Recovered magnetic moment of the fitted cell
    #   m = M * V        (scalar, A.m^2)
    #   m_vec = (Mx,My,Mz) * V   (vector components, A.m^2)
    # ------------------------------------------------------------
    df["Moment"] = df["Mag_norm"] * df["Volume"]
    df["mx"] = df["Mx"] * df["Volume"]
    df["my"] = df["My"] * df["Volume"]
    df["mz"] = df["Mz"] * df["Volume"]

    cols = ["CellID", "i", "j", "k", "X1", "X2", "Y1", "Y2", "Z1", "Z2",
            "dx", "dy", "dz", "Volume", "Mx", "My", "Mz",
            "mx", "my", "mz",
            "Mag_norm", "Dec", "Inc", "Moment"]
    nz_out = df.loc[df["Mag_norm"] > 0, cols].copy()

    n = len(nz_out)
    if n == 0:
        print("   Fitted cell has zero magnetization - skipping save.")
        continue

    # ------------------------------------------------------------
    # With a single fitted cell, the "bulk" quantities collapse to
    # that one cell's own direction/magnitude/moment -- no Fisher
    # statistics needed (N=1 cell), but kept in the same summary
    # layout as the full-model script for consistency downstream.
    # ------------------------------------------------------------
    fitted_dec  = nz_out["Dec"].iloc[0]
    fitted_inc  = nz_out["Inc"].iloc[0]
    fitted_mag  = nz_out["Mag_norm"].iloc[0]
    fitted_moment_vec = np.array([nz_out["mx"].iloc[0], nz_out["my"].iloc[0], nz_out["mz"].iloc[0]])

    summary = {
        "Fitted_Dec":                  fitted_dec,
        "Fitted_Inc":                  fitted_inc,
        "Fitted_Mag":                  fitted_mag,
        "Fitted_mx":                   fitted_moment_vec[0],
        "Fitted_my":                   fitted_moment_vec[1],
        "Fitted_mz":                   fitted_moment_vec[2],
        "Fitted_moment_magnitude":     np.linalg.norm(fitted_moment_vec),
        "Fitted_Volume_m3":            nz_out["Volume"].iloc[0],
    }

    print(f"   N_cells (fitted) = {n}")
    for key, val in summary.items():
        print(f"   {key:22s}: {val:,.25f}")

    for key in summary:
        nz_out[key] = np.nan
    nz_out.loc[nz_out.index[0], list(summary.keys())] = list(summary.values())

    out_path = os.path.join(body_folder_Out, f"{body}_fitted.csv")
    nz_out.to_csv(out_path, index=False)
    print("   Saved:", out_path)

print("\nDONE.")

### 5- 3D plot voxels

### 6- recovered misfits plots with heights

In [ ]:
#against height
%matplotlib inline

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.ticker import LogLocator, FuncFormatter

# ------------------------------------------------------------
# PATHS
#
# True moment still comes from the forward (full-mesh) model, as
# before. The "recovered" side now comes from the FITTED
# single-cell model (Tomofast_Output_Fitting_Steepest/{body}_fitted.csv,
# columns Fitted_mx/my/mz, produced by analyze_fitted_cell.py) instead
# of the full-grid inversion output (Tomofast_Output_inversion).
# ------------------------------------------------------------
base_dir = os.getcwd()
FORWARD_ROOT = os.path.join(base_dir, "Tec_files_bellon", "converted", "tomofast_forward")
FITTED_ROOT  = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_Fitting_Steepest")
METRICS_CSV  = os.path.join(FITTED_ROOT, "inversion_metrics.csv")  # optional -- see note below
GRAIN_CSV    = r"C:\Users\23121159\OneDrive - Curtin\Ongoing\JGR_Review\Work\1-Merrill\Particle_morphology_data\Grain_size_distirbution.csv"

plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      13,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "pdf.fonttype":   42,
    "ps.fonttype":    42,
})

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def extract_grain_id(body_name):
    """body_118_24_mult_h0.010 -> '118' (unaffected by the trailing height tag)"""
    return body_name.split("_")[1]


def extract_height_um(body_name):
    """
    body_118_24_mult_h0.010 -> 0.010 (float, same units SENSOR_HEIGHT used,
    µm). Reads the trailing '_h<number>' tag added by tec_converter.py's
    height-sweep naming convention. Returns np.nan if no tag is found
    (e.g. an older body folder generated before the height sweep existed).
    """
    m = re.search(r"_h([0-9]*\.?[0-9]+)$", body_name)
    if m is None:
        return np.nan
    return float(m.group(1))


def r2_to_size(r2):
    if np.isnan(r2):
        return 60
    if r2 > 0.95:
        return 120
    elif r2 > 0.85:
        return 60
    else:
        return 20


def vec_to_dec_inc(Mx, My, Mz):
    """
    Vector -> (Dec, Inc) in degrees, using the same convention as the rest
    of the pipeline (Mz flipped internally, Inc = theta-90, Dec = 90-phi).
    """
    norm = np.sqrt(Mx ** 2 + My ** 2 + Mz ** 2)
    theta = np.degrees(np.arccos(np.clip(-Mz / norm, -1, 1)))
    phi = np.degrees(np.arctan2(My, Mx))
    inc = theta - 90.0
    dec = np.mod(90.0 - phi, 360.0)
    return dec, inc


def circular_diff_deg(a, b):
    """Shortest angular distance between two angles in degrees, range [0, 180]."""
    return abs((b - a + 180) % 360 - 180)


def decimal_formatter(val, pos):
    """Format log-scale tick labels as plain decimals (0.01, 0.001, ...)
    instead of scientific notation (1e-2, 1e-3, ...)."""
    if val == 0:
        return "0"
    s = f"{val:.10f}".rstrip("0").rstrip(".")
    return s


# ------------------------------------------------------------
# LOAD grain size (Emax)
# body column in CSV is just the number: 9, 12, 14 ...
# ------------------------------------------------------------
grain_df = pd.read_csv(GRAIN_CSV)
grain_df["body_key"] = grain_df["body"].astype(str)
grain_df = grain_df.set_index("body_key")

# ------------------------------------------------------------
# LOAD R2 from metrics CSV, IF it exists.
#
# NOTE: the fitting workflow doesn't currently produce a per-grain R2
# misfit-vs-data file the way the full inversion's run does (that
# came from a separate inversion-quality step we haven't built for
# the single-cell fitting pipeline). If the metrics file isn't
# found, R2 is left as NaN for every grain, which r2_to_size() already
# maps to a default marker size -- so the plot still runs, just
# without that visual encoding, instead of crashing.
# ------------------------------------------------------------
if os.path.exists(METRICS_CSV):
    metrics_df = pd.read_csv(METRICS_CSV)
    if "component" in metrics_df.columns:
        metrics_df = metrics_df[metrics_df["component"] == "Bz"].copy()
    metrics_df = metrics_df.set_index("inversion")
    have_metrics = True
else:
    print(f"[NOTE] {METRICS_CSV} not found -- R2 will be NaN for all grains "
          f"(marker size will use the default tier).")
    metrics_df = None
    have_metrics = False

# ------------------------------------------------------------
# LOAD true (forward) and fitted moments, compute misfits
# ------------------------------------------------------------
body_dirs = sorted(
    d for d in os.listdir(FORWARD_ROOT)
    if d.startswith("body_") and os.path.isdir(os.path.join(FORWARD_ROOT, d))
)

records = []

for body in body_dirs:
    fwd_csv = glob.glob(os.path.join(FORWARD_ROOT, body, f"{body}_nonzero.csv"))
    fit_csv = glob.glob(os.path.join(FITTED_ROOT, body, f"{body}_fitted.csv"))

    if not fwd_csv or not fit_csv:
        print(f"Skipping {body}: missing CSV.")
        if not fit_csv:
            print(f"  -> expected: {os.path.join(FITTED_ROOT, body, f'{body}_fitted.csv')} "
                  f"(run analyze_fitted_cell.py first)")
        continue

    sensor_height = extract_height_um(body)
    if np.isnan(sensor_height):
        print(f"Skipping {body}: could not parse a sensor height from the body name "
              f"(expected a trailing '_h<value>' tag, e.g. '..._h0.010').")
        continue

    fwd = pd.read_csv(fwd_csv[0], nrows=1)
    fit = pd.read_csv(fit_csv[0], nrows=1)

    m_true = np.array([
        fwd["True_mx"].values[0],
        fwd["True_my"].values[0],
        fwd["True_mz"].values[0],
    ])
    m_fit = np.array([
        fit["Fitted_mx"].values[0],
        fit["Fitted_my"].values[0],
        fit["Fitted_mz"].values[0],
    ])

    norm_true = np.linalg.norm(m_true)
    norm_fit  = np.linalg.norm(m_fit)

    if norm_true == 0 or norm_fit == 0:
        print(f"Skipping {body}: zero moment.")
        continue

    # Angular misfit (true 3D angle between the two moment vectors)
    cos_angle      = np.dot(m_true, m_fit) / (norm_true * norm_fit)
    angular_misfit = np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))

    # Amplitude misfit (%)
    amplitude_misfit = abs(norm_fit - norm_true) / norm_true * 100

    # Dec/Inc of each vector, and their individual angular differences
    dec_true, inc_true = vec_to_dec_inc(*m_true)
    dec_fit, inc_fit   = vec_to_dec_inc(*m_fit)
    angular_dec = circular_diff_deg(dec_true, dec_fit)   # wraps correctly at 0/360
    angular_inc = abs(inc_fit - inc_true)                # inc has no wraparound (-90..90)

    # Emax lookup: extract grain number from body name
    grain_key = extract_grain_id(body)
    emax = grain_df.loc[grain_key, "Max_Elongation"] if grain_key in grain_df.index else np.nan
    if np.isnan(emax):
        print(f"  Warning: no Emax found for {body} (grain_key={grain_key})")

    # R2 lookup (NaN if no metrics file -- see note above)
    r2 = metrics_df.loc[body, "r2"] if (have_metrics and body in metrics_df.index) else np.nan

    records.append({
        "body":             body,
        "grain_key":        grain_key,
        "sensor_height":    sensor_height,
        "Angular_misfit":   angular_misfit,
        "Amplitude_misfit": amplitude_misfit,
        "Angular_Dec":      angular_dec,
        "Angular_Inc":      angular_inc,
        "Dec_true":         dec_true,
        "Inc_true":         inc_true,
        "Dec_fit":          dec_fit,
        "Inc_fit":          inc_fit,
        "Emax":             emax,
        "R2":               r2,
    })

df_results = pd.DataFrame(records)
print(df_results.to_string())

df_results.to_csv(os.path.join(FITTED_ROOT, "misfit_results_fitted.csv"), index=False)

# ------------------------------------------------------------
# Build color/marker map from the EXACT distinct Emax values present
# in the data (not fixed bins/ranges) -- each unique Emax value gets
# its own color and marker shape.
# ------------------------------------------------------------
unique_emax = np.sort(df_results["Emax"].dropna().unique())

_marker_cycle = ["o", "s", "^", "v", "D", "P", "X", "*", "<", ">", "h", "8"]
_color_cycle  = plt.cm.tab10.colors  # 10 distinct colors, cycles if more values than that

emax_color_map  = {val: _color_cycle[i % len(_color_cycle)] for i, val in enumerate(unique_emax)}
emax_marker_map = {val: _marker_cycle[i % len(_marker_cycle)] for i, val in enumerate(unique_emax)}

# ------------------------------------------------------------
# PLOT -- styled to match the published figure: log-log axes,
# fixed y-range with explicit decade ticks shown as plain decimals,
# full flat shading below tolerance (not just a thin band), dashed
# lines at the actual swept heights present in the data, and legend
# entries keyed to each exact Emax value (not ranges).
# ------------------------------------------------------------
TOLERANCE_ANG = 5.0
TOLERANCE_AMP = 10.0   # matches the paper figure's shading boundary

def auto_log_ylim_ticks(series, tolerance, pad_decades=0.5):
    """Compute y-limits/ticks spanning just the actual data range (plus the
    tolerance line and a little padding), instead of a fixed wide range."""
    vals = series.replace(0, np.nan).dropna()
    vals = vals[vals > 0]
    if len(vals) == 0:
        lo, hi = tolerance / 10, tolerance * 10
    else:
        lo = min(vals.min(), tolerance)
        hi = max(vals.max(), tolerance)
    log_lo = np.floor(np.log10(lo) - pad_decades)
    log_hi = np.ceil(np.log10(hi) + pad_decades)
    ylim = (10 ** log_lo, 10 ** log_hi)
    yticks = [10 ** e for e in range(int(log_hi), int(log_lo) - 1, -1)]
    return ylim, yticks

YLIM_ANG, YTICKS_ANG = auto_log_ylim_ticks(df_results["Angular_misfit"], TOLERANCE_ANG)
YLIM_AMP, YTICKS_AMP = auto_log_ylim_ticks(df_results["Amplitude_misfit"], TOLERANCE_AMP)

PANELS = [
    ("Angular_misfit",   "Angular misfit (°)",     "Single Cell Inversion",      TOLERANCE_ANG, YLIM_ANG, YTICKS_ANG),
    ("Amplitude_misfit", "Amplitude misfit (%)",   "Single Cell Inversion",    TOLERANCE_AMP, YLIM_AMP, YTICKS_AMP),
    ("Angular_Dec",      "Declination misfit (°)", "Single Cell Inversion", TOLERANCE_ANG, YLIM_ANG, YTICKS_ANG),
    ("Angular_Inc",      "Inclination misfit (°)", "Single Cell Inversion", TOLERANCE_ANG, YLIM_ANG, YTICKS_ANG),
]

# reference heights actually swept (not hardcoded) -- drawn as dashed
# vertical lines with rotated labels, same as the paper figure
all_heights = np.sort(df_results["sensor_height"].dropna().unique())
xmin = all_heights.min() / 1.5 if len(all_heights) else 0.005
xmax = all_heights.max() * 1.5 if len(all_heights) else 50

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (column, ylabel, title, tolerance, ylim, yticks) in zip(axes, PANELS):
    # full flat shaded "acceptable fit" region below tolerance
    ax.axhspan(ylim[0], tolerance, color="#e6e6e6", zorder=0)

    # connect points from the same grain across heights with a line,
    # so you can trace how each grain's misfit moves as height changes
    for grain_key, grp in df_results.groupby("grain_key"):
        grp = grp.sort_values("sensor_height")
        if len(grp) < 2:
            continue
        emax_val = grp["Emax"].iloc[0]
        line_col = emax_color_map.get(emax_val, "gray")
        ax.plot(grp["sensor_height"], grp[column], color=line_col,
                linewidth=1.0, alpha=0.6, zorder=2)

    for _, row in df_results.iterrows():
        emax = row["Emax"]
        r2   = row["R2"]

        if np.isnan(emax):
            col, mark = "gray", "o"
        else:
            col, mark = emax_color_map[emax], emax_marker_map[emax]

        size = r2_to_size(r2)
        ax.scatter(row["sensor_height"], row[column], facecolor=col, edgecolors=col,
                   marker=mark, s=size, linewidths=1.0, zorder=3)

    # dashed reference height lines + rotated labels (actual sweep, not hardcoded)
    for h in all_heights:
        ax.axvline(h, color="black", linestyle="--", linewidth=2.2, zorder=2)
        ax.text(h, ylim[1] * 0.55, f"{h:.2f} \u00b5m", rotation=90,
                va="top", ha="center", fontsize=8)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(*ylim)
    ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=20))
    ax.set_yticks(yticks)
    ax.yaxis.set_major_formatter(FuncFormatter(decimal_formatter))
    ax.xaxis.set_major_formatter(FuncFormatter(decimal_formatter))
    ax.set_xlabel("Source to Sensor distance (µm)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(False)

# legend: exact Emax values, each with its own color/marker
legend_handles_emax = [
    plt.scatter([], [], facecolor=emax_color_map[val], edgecolor=emax_color_map[val], marker=emax_marker_map[val],
                s=80, linewidths=1.0, label=fr"$E_{{\max}} = {val:.2f}\ \mu m$")
    for val in unique_emax
]

legend_handles = legend_handles_emax
if have_metrics:
    # legend: R2 size tiers (only meaningful if we actually have R2 values)
    legend_handles_r2 = [
        plt.scatter([], [], facecolor="none", edgecolor="black", marker="o", s=120,
                    linewidths=1.0, label=r"$R^2 > 0.95$"),
        plt.scatter([], [], facecolor="none", edgecolor="black", marker="o", s=60,
                    linewidths=1.0, label=r"$0.85 < R^2 \leq 0.95$"),
        plt.scatter([], [], facecolor="none", edgecolor="black", marker="o", s=20,
                    linewidths=1.0, label=r"$R^2 \leq 0.85$"),
    ]
    legend_handles = legend_handles_emax + legend_handles_r2

axes[0].legend(handles=legend_handles, fontsize=9, loc="lower left", framealpha=0.95)

plt.tight_layout()
plt.savefig(os.path.join(FITTED_ROOT, "misfit_per_grain_fitted.png"), dpi=400, bbox_inches="tight")
plt.savefig(os.path.join(FITTED_ROOT, "misfit_per_grain_fitted.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# R2 vs sensor height - FITTING (Steepest) version
# Plots R²_dip against sensor height extracted from body folder names,
# colour/marker-coded by EXACT Emax value (not bins), with each grain's
# points connected by a line across heights so you can trace how R²
# moves as height changes. R² comes from the optional metrics CSV;
# if absent all markers use the default grey tier (same graceful fallback
# as the original fitting script).
%matplotlib inline

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
base_dir     = os.getcwd()
FORWARD_ROOT = os.path.join(base_dir, "Tec_files_bellon", "converted", "tomofast_forward")
FITTED_ROOT  = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_Fitting_Steepest")
METRICS_CSV  = os.path.join(FITTED_ROOT, "inversion_metrics.csv")   # optional
GRAIN_CSV    = r"C:\Users\23121159\OneDrive - Curtin\Ongoing\JGR_Review\Work\1-Merrill\Particle_morphology_data\Grain_size_distirbution.csv"

plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      13,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "pdf.fonttype":   42,
    "ps.fonttype":    42,
})

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def extract_grain_id(body_name):
    """body_118_24_mult_h0.010 -> '118' (unaffected by the trailing height tag)"""
    return body_name.split("_")[1]


def extract_height_um(body_name):
    m = re.search(r"_h([0-9]*\.?[0-9]+)$", body_name)
    return float(m.group(1)) if m else np.nan


def decimal_formatter(val, pos):
    """Format log-scale tick labels as plain decimals (0.01, 0.001, ...)
    instead of scientific notation (1e-2, 1e-3, ...)."""
    if val == 0:
        return "0"
    s = f"{val:.10f}".rstrip("0").rstrip(".")
    return s


# ------------------------------------------------------------
# LOAD grain size (Emax)
# ------------------------------------------------------------
grain_df = pd.read_csv(GRAIN_CSV)
grain_df["body_key"] = grain_df["body"].astype(str)
grain_df = grain_df.set_index("body_key")

# ------------------------------------------------------------
# LOAD R² (optional)
# ------------------------------------------------------------
if os.path.exists(METRICS_CSV):
    metrics_df = pd.read_csv(METRICS_CSV)
    if "component" in metrics_df.columns:
        metrics_df = metrics_df[metrics_df["component"] == "Bz"].copy()
    metrics_df = metrics_df.set_index("inversion")
    have_metrics = True
    print(f"Loaded metrics from {METRICS_CSV}")
else:
    print(f"[NOTE] {METRICS_CSV} not found - R² will be NaN for all grains.")
    metrics_df   = None
    have_metrics = False

# ------------------------------------------------------------
# BUILD records
# ------------------------------------------------------------
body_dirs = sorted(
    d for d in os.listdir(FORWARD_ROOT)
    if d.startswith("body_") and os.path.isdir(os.path.join(FORWARD_ROOT, d))
)

records = []
for body in body_dirs:
    # We only need bodies that also have a fitted CSV
    fit_csv = glob.glob(os.path.join(FITTED_ROOT, body, f"{body}_fitted.csv"))
    if not fit_csv:
        print(f"Skipping {body}: no fitted CSV "
              f"(run analyze_fitted_cell.py first).")
        continue

    sensor_height = extract_height_um(body)
    if np.isnan(sensor_height):
        print(f"Skipping {body}: no height tag.")
        continue

    r2 = (metrics_df.loc[body, "r2"]
          if (have_metrics and body in metrics_df.index) else np.nan)

    grain_key = extract_grain_id(body)
    emax = (grain_df.loc[grain_key, "Max_Elongation"]
            if grain_key in grain_df.index else np.nan)

    records.append({
        "body":          body,
        "grain_key":     grain_key,
        "sensor_height": sensor_height,
        "R2":            r2,
        "Emax":          emax,
    })

df = pd.DataFrame(records)
print(df.to_string())
df.to_csv(os.path.join(FITTED_ROOT, "r2_vs_height_fitted.csv"), index=False)

# ------------------------------------------------------------
# Build color/marker map from the EXACT distinct Emax values present
# in the data (not fixed bins/ranges) -- each unique Emax value gets
# its own color and filled marker shape, matching the misfit plots.
# ------------------------------------------------------------
unique_emax = np.sort(df["Emax"].dropna().unique())

_marker_cycle = ["o", "s", "^", "v", "D", "P", "X", "*", "<", ">", "h", "8"]
_color_cycle  = plt.cm.tab10.colors  # 10 distinct colors, cycles if more values than that

emax_color_map  = {val: _color_cycle[i % len(_color_cycle)] for i, val in enumerate(unique_emax)}
emax_marker_map = {val: _marker_cycle[i % len(_marker_cycle)] for i, val in enumerate(unique_emax)}

heights_sorted = sorted(df["sensor_height"].unique())

# ------------------------------------------------------------
# PLOT
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 5))

ax.axhline(0.90, color="black", linestyle="--", linewidth=1.0, zorder=1)
ax.axhline(0.50, color="black", linestyle="--", linewidth=1.0, zorder=1)

# connect points from the same grain across heights with a line,
# so you can trace how each grain's R2 moves as height changes
for grain_key, grp in df.groupby("grain_key"):
    grp = grp.dropna(subset=["R2"]).sort_values("sensor_height")
    if len(grp) < 2:
        continue
    emax_val = grp["Emax"].iloc[0]
    line_col = emax_color_map.get(emax_val, "gray")
    ax.plot(grp["sensor_height"], grp["R2"], color=line_col,
            linewidth=1.0, alpha=0.6, zorder=2)

for _, row in df.iterrows():
    if np.isnan(row["R2"]):
        continue
    emax = row["Emax"]
    if np.isnan(emax):
        col, mark = "gray", "o"
    else:
        col, mark = emax_color_map[emax], emax_marker_map[emax]

    ax.scatter(row["sensor_height"], row["R2"], facecolor=col, edgecolors=col,
               marker=mark, s=60, linewidths=1.0, zorder=3)

ax.set_xscale("log")
ax.xaxis.set_major_formatter(FuncFormatter(decimal_formatter))
ax.set_xlabel("Source to Sensor distance (µm)")
ax.set_ylabel(r"$R^2_{\mathrm{fitting}}$")
ax.set_title("(a) Fitting (Steepest) - $R^2$ vs sensor height")
ax.set_xlim(left=min(heights_sorted) / 1.5, right=max(heights_sorted) * 1.5)
ax.set_ylim(-1.0, 1.05)
ax.grid(True, axis="y", linewidth=0.4, alpha=0.6)
ax.grid(False, axis="x")

# legend: exact Emax values, each with its own filled color/marker
legend_handles = [
    plt.scatter([], [], facecolor=emax_color_map[val], edgecolor=emax_color_map[val],
                marker=emax_marker_map[val], s=80, linewidths=1.0,
                label=fr"$E_{{\max}} = {val:.2f}\ \mu m$")
    for val in unique_emax
]
ax.legend(handles=legend_handles, title=r"$E_{\max}$", fontsize=10,
          title_fontsize=10, loc="lower right", framealpha=0.9)

plt.tight_layout()
plt.savefig(os.path.join(FITTED_ROOT, "r2_vs_height_fitted.png"), dpi=400, bbox_inches="tight")
plt.savefig(os.path.join(FITTED_ROOT, "r2_vs_height_fitted.pdf"), bbox_inches="tight")
plt.show()